In [2]:
# Entrenar un modelo simple para clasificar imágenes de ropa y comparar el
# tiempo que tarda en entrenarse en CPU vs. GPU.

# --- 1. Importar librerías ---
import tensorflow as tf
import time
from tensorflow.keras.datasets import fashion_mnist

#1. Importar la primera (1) base de datos para operar...
print("TensorFlow version:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))


#Imágenes que debo transformar
#Nota: las imágenes están conformadas por pixeles de diferentes tipos de gris. Eso es lo que
#se necesita procesar. El modelo con la data procesada le permite leer la distribución de los
#pixeles para detectar la forma y asignarlo a un producto (zapato, vestido, etc). Aquí el
#objetivo.


#2.1 Normalizar para mejor manipulación de data:

(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
X_train = X_train / 255.0
X_test = X_test / 255.0

# 3. Entrenamiento en CPU
def crear_modelo():
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(28, 28)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# 4. Entrenamiento en GPU (solo si tienes GPU disponible)
print("\n--- Entrenamiento en CPU ---")
with tf.device('/CPU:0'):
    modelo_cpu = crear_modelo()
    start_cpu = time.time()
    modelo_cpu.fit(X_train, y_train, epochs=5, verbose=2)
    end_cpu = time.time()
    print("Tiempo en CPU:", round(end_cpu - start_cpu, 2), "segundos")

#5. Entrenamiento en GPU
if tf.config.list_physical_devices('GPU'):
    print("\n--- Entrenamiento en GPU ---")
    modelo_gpu = crear_modelo()
    with tf.device('/GPU:0'):
        start_gpu = time.time()
        modelo_gpu.fit(X_train, y_train, epochs=5, verbose=2)
        end_gpu = time.time()
        print("Tiempo en GPU:", round(end_gpu - start_gpu, 2), "segundos")
else:
    print(" No se detectó GPU.")


TensorFlow version: 2.18.0
GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- Entrenamiento en CPU ---
Epoch 1/5
1875/1875 - 5s - 3ms/step - accuracy: 0.8270 - loss: 0.4931
Epoch 2/5
1875/1875 - 5s - 3ms/step - accuracy: 0.8656 - loss: 0.3722
Epoch 3/5
1875/1875 - 4s - 2ms/step - accuracy: 0.8777 - loss: 0.3344
Epoch 4/5
1875/1875 - 4s - 2ms/step - accuracy: 0.8857 - loss: 0.3129
Epoch 5/5
1875/1875 - 5s - 3ms/step - accuracy: 0.8905 - loss: 0.2935
⏱️ Tiempo en CPU: 25.15 segundos

--- Entrenamiento en GPU ---
Epoch 1/5
1875/1875 - 6s - 3ms/step - accuracy: 0.8276 - loss: 0.4947
Epoch 2/5
1875/1875 - 9s - 5ms/step - accuracy: 0.8651 - loss: 0.3752
Epoch 3/5
1875/1875 - 4s - 2ms/step - accuracy: 0.8765 - loss: 0.3364
Epoch 4/5
1875/1875 - 4s - 2ms/step - accuracy: 0

In [1]:
#Parte 2: Toxic Comments

# 1. Dataset Hugging Face

from google.colab import files
uploaded = files.upload()

# 2. Cargar el csv con Pandas y ponerlo en formato Hugging Face para seguir con el formato de la tarea

import pandas as pd

# Carga el archivo CSV
df = pd.read_csv("train.csv")

# Verifica las primeras filas
df.head()




Saving train.csv to train.csv


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [2]:
# 3. Preparar el dataset con la clasificación de mensajes:

    # Convertir las etiquetas: 1 si toxicidad ≥ 0.5, 0 si < 0.5
df['toxic'] = (df['toxic'] >= 0.5).astype(int)

    # Eliminar filas con comentarios vacíos (por si acaso)
df = df[df['comment_text'].notnull()]

    # Mostrar muestra
df[['comment_text', 'toxic']].head()


,comment_text,toxic
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0


In [7]:
df['toxic'].value_counts()


,count
toxic,
0,144277
1,15294


In [3]:
# 4. Tokenizar

# Por si acaso, descargo librerías...
!pip install transformers datasets --quiet
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')    # el de las indicaciones

# Aplico el tokenizer al subconjunto de datos para entrenar
from datasets import Dataset
import pandas as pd

# archivo ya subido y preprocesado (10,000 filas para que corra)
df = pd.read_csv("train.csv")[["comment_text", "toxic"]].dropna()
df["label"] = (df["toxic"] >= 0.5).astype(int)
df = df[["comment_text", "label"]]
df_small = df.sample(n=10000, random_state=42).reset_index(drop=True)

# Convertimos a formato Hugging Face
dataset = Dataset.from_pandas(df_small)

# Tokenizamos el texto
def tokenize_function(example):
    return tokenizer(example["comment_text"], padding="max_length", truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# 5. Design binary model...
from transformers import AutoModelForSequenceClassification

# Modelo BERT para clasificación binaria (0 = no tóxico, 1 = tóxico)
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
print(f"Número total de ejemplos: {len(df)}")


Número total de ejemplos: 159571


In [4]:
# Argumentos para el entrenamiento:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_cpu",     # Carpeta donde se guardan los resultados
    num_train_epochs=2,          # Número de epochs
    per_device_train_batch_size=8,    # Tamaño del batch
    logging_dir="./logs",                 # Carpeta de logs
    logging_steps=10,                     # número de pasos en que se muestra log
)


In [5]:
# 6. Cargar modelo - ENTRENAMIENTO

from transformers import BertForSequenceClassification

# Modelo BERT base para clasificación binaria (2 etiquetas: tóxico o no tóxico)
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

## DEF. DEL ENTRENAMIENTO

from transformers import Trainer, TrainingArguments
from transformers import DataCollatorWithPadding

# Asegúrate de que `tokenized_dataset` tenga la columna 'label'
# Dividir el dataset en entrenamiento y evaluación
train_test = tokenized_dataset.train_test_split(test_size=0.2)

# Padding dinámico
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_test["train"],
    eval_dataset=train_test["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-5-3835496030.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [13]:
## CPU VS GPU

import time

start = time.time()
trainer.train()
end = time.time()

print(f"Tiempo total de entrenamiento en CPU: {end - start:.2f} segundos")

#a0973a56efed0646f40b3252716b55efe65c8d79

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: eschavezalvarado (eschavezalvarado-universidad-del-pac-fico) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


KeyboardInterrupt: 

In [6]:
import time

start = time.time()
trainer.train()
end = time.time()

print(f"Tiempo total de entrenamiento en GPU: {end - start:.2f} segundos")


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: eschavezalvarado (eschavezalvarado-universidad-del-pac-fico) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,0.459200
20,0.245700
30,0.313700
40,0.224700
50,0.371300
60,0.273700
70,0.276500
80,0.216100
90,0.357100
100,0.146000


KeyboardInterrupt: 